# G13 (31B): temperature probe, arm e4b_b2, e4b_v on all 75 competition tasks

Base reference at temperature 0.2: r1 25/75, r2 28/75 (+ adapters 28, 22). Output: `g13/g13_summary.json` (config gates + mechanism summary), `g13/capture.jsonl.gz`, `g13/t4/`.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

## Setup

In [ ]:
import json, os, random, shutil, sys, threading, time, traceback
from pathlib import Path
WORKING_DIR = Path('/kaggle/working'); WORKING_DIR.mkdir(parents=True, exist_ok=True)
G2 = WORKING_DIR / 'g14c'; G2.mkdir(exist_ok=True); CODE = G2 / 'code'; CODE.mkdir(exist_ok=True)
RESULTS = {'started': time.strftime('%Y-%m-%d %H:%M:%S')}
import torch
_gpus = [(torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]
print('GPUs:', _gpus)
assert len(_gpus) == 4 and all(c[0] >= 8 for _, c in _gpus), f'expected 4x L4 (compute capability >= 8), got {_gpus}'
def save_results(): (G2 / 'g14c_summary.json').write_text(json.dumps(RESULTS, indent=1, default=str))
BUDGET_MIN = 200; T_START = time.time()
def _watchdog():   # a hung round must not burn GPU quota
    RESULTS['watchdog'] = f'fired after {BUDGET_MIN} min'; save_results(); os._exit(3)
_wd = threading.Timer(BUDGET_MIN * 60, _watchdog); _wd.daemon = True; _wd.start()
for n, txt in {"capture_proxy.py": "\"\"\"Transparent logging reverse proxy between the harness (LiteLLM -> ADK) and vLLM, for G2 data collection.\n\nFor every non-streaming POST .../chat/completions it records, gzip-compressed, one JSON line with the FULL request body (messages incl. reasoning,\ntools, chat_template_kwargs, sampling params) and the FULL response body. It asks vLLM for `return_token_ids` so the record also carries the exact\nprompt token ids and completion token ids (no re-rendering guesses); if the server rejects that parameter it retries without it and keeps going.\nThe harness receives the response with the token-id fields stripped, i.e. exactly what it would get without the proxy.\nEverything else (GET /v1/models, /health, streaming requests) is passed through untouched.\n\"\"\"\n\nimport gzip\nimport hashlib\nimport json\nimport threading\nimport time\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\nimport requests\n\n\ndef task_key(messages: list[dict]) -> str:\n    \"\"\"Stable id of the agent session a request belongs to: hash of the system prompt + the first user message (the task statement).\"\"\"\n    sys_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"system\"), \"\") or \"\"\n    usr_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"user\"), \"\") or \"\"\n    h = hashlib.sha1()\n    for part in (sys_txt, usr_txt):\n        h.update((part if isinstance(part, str) else json.dumps(part, sort_keys=True)).encode(\"utf-8\", \"ignore\"))\n        h.update(b\"\\x00\")\n    return h.hexdigest()[:16]\n\n\nclass CaptureProxy:\n    def __init__(self, upstream: str, out_path: str, port: int = 8001, host: str = \"127.0.0.1\", ask_token_ids: bool = True):\n        self.upstream = upstream.rstrip(\"/\")          # e.g. http://127.0.0.1:8000\n        self.out_path = out_path\n        self.port, self.host = port, host\n        self.ask_token_ids = ask_token_ids\n        self.round_tag = \"r0\"\n        self.tokenize_fallback = False        # if True and vLLM returns no token ids, ask /tokenize for the exact prompt ids instead\n        self.n_records = 0\n        self.n_with_ids = 0\n        self.n_errors = 0\n        self._lock = threading.Lock()\n        self._httpd: ThreadingHTTPServer | None = None\n\n    # ---------------------------------------------------------------- lifecycle\n    def start(self) -> \"CaptureProxy\":\n        proxy = self\n\n        class H(BaseHTTPRequestHandler):\n            protocol_version = \"HTTP/1.1\"\n\n            def log_message(self, *a):  # silence\n                pass\n\n            def _relay(self, status: int, body: bytes, ctype: str = \"application/json\"):\n                self.send_response(status)\n                self.send_header(\"Content-Type\", ctype)\n                self.send_header(\"Content-Length\", str(len(body)))\n                self.end_headers()\n                self.wfile.write(body)\n\n            def do_GET(self):\n                try:\n                    r = requests.get(proxy.upstream + self.path, timeout=60)\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                except Exception as e:  # noqa: BLE001\n                    self._relay(502, json.dumps({\"error\": str(e)}).encode())\n\n            def do_POST(self):\n                raw = self.rfile.read(int(self.headers.get(\"Content-Length\", \"0\")))\n                try:\n                    req = json.loads(raw) if raw else {}\n                except Exception:  # noqa: BLE001\n                    req = None\n                is_chat = self.path.rstrip(\"/\").endswith(\"/chat/completions\") and isinstance(req, dict) and not req.get(\"stream\")\n                if not is_chat:                      # pass-through (streaming, other endpoints)\n                    try:\n                        r = requests.post(proxy.upstream + self.path, data=raw, headers={\"Content-Type\": \"application/json\"}, timeout=1800)\n                        self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                    except Exception as e:  # noqa: BLE001\n                        self._relay(502, json.dumps({\"error\": str(e)}).encode())\n                    return\n                t0 = time.time()\n                asked = proxy.ask_token_ids           # decided ONCE per request (another thread may flip the flag meanwhile)\n                body = dict(req, return_token_ids=True) if asked else req\n                try:\n                    r = requests.post(proxy.upstream + self.path, json=body, timeout=1800)\n                    if asked and r.status_code == 400 and \"return_token_ids\" in r.text:\n                        proxy.ask_token_ids = False   # this vLLM rejects the parameter: stop asking, redo this call plainly\n                        r = requests.post(proxy.upstream + self.path, json=req, timeout=1800)\n                except Exception as e:  # noqa: BLE001\n                    proxy.n_errors += 1\n                    self._relay(502, json.dumps({\"error\": f\"proxy upstream failure: {e}\"}).encode())\n                    return\n                try:\n                    resp = r.json()\n                except Exception:  # noqa: BLE001\n                    resp = None\n                if r.status_code == 200 and isinstance(resp, dict):\n                    proxy._record(req, resp, t0, time.time())\n                    clean = {k: v for k, v in resp.items() if k != \"prompt_token_ids\"}\n                    clean[\"choices\"] = [{k: v for k, v in ch.items() if k != \"token_ids\"} for ch in resp.get(\"choices\", [])]\n                    self._relay(200, json.dumps(clean).encode())\n                else:\n                    proxy.n_errors += 1\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n\n        self._httpd = ThreadingHTTPServer((self.host, self.port), H)\n        self._httpd.daemon_threads = True\n        threading.Thread(target=self._httpd.serve_forever, daemon=True).start()\n        return self\n\n    def stop(self) -> None:\n        if self._httpd:\n            self._httpd.shutdown()\n            self._httpd.server_close()\n\n    # ---------------------------------------------------------------- recording\n    def _record(self, req: dict, resp: dict, t0: float, t1: float) -> None:\n        has_ids = bool(resp.get(\"prompt_token_ids\")) and bool((resp.get(\"choices\") or [{}])[0].get(\"token_ids\"))\n        rec = {\"round\": self.round_tag, \"task_key\": task_key(req.get(\"messages\", [])), \"t0\": round(t0, 3), \"t1\": round(t1, 3), \"request\": req, \"response\": resp}\n        if not has_ids and self.tokenize_fallback:\n            try:\n                tk = {k: req[k] for k in (\"model\", \"messages\", \"tools\", \"chat_template_kwargs\") if k in req}\n                tk[\"add_generation_prompt\"] = True\n                r = requests.post(self.upstream + \"/tokenize\", json=tk, timeout=300)\n                if r.status_code == 200:\n                    rec[\"prompt_token_ids_tokenize\"] = r.json().get(\"tokens\")\n            except Exception:  # noqa: BLE001\n                pass\n        line = (json.dumps(rec, ensure_ascii=False) + \"\\n\").encode(\"utf-8\")\n        with self._lock:                             # one gzip member per record; the concatenation is a valid gzip stream\n            with open(self.out_path, \"ab\") as f:\n                f.write(gzip.compress(line, compresslevel=3))\n            self.n_records += 1\n            self.n_with_ids += has_ids\n\n    def stats(self) -> dict:\n        return {\"records\": self.n_records, \"with_token_ids\": self.n_with_ids, \"errors\": self.n_errors, \"asking_token_ids\": self.ask_token_ids, \"round\": self.round_tag}\n\n\ndef read_records(path: str):\n    \"\"\"Iterate the records of a capture file.\"\"\"\n    with gzip.open(path, \"rt\", encoding=\"utf-8\") as f:\n        for line in f:\n            if line.strip():\n                yield json.loads(line)\n"}.items(): (CODE / n).write_text(txt, encoding='utf-8')
sys.path.insert(0, str(CODE))
DATA_DIR = next(p.parent for p in Path('/kaggle/input').rglob('tasks.jsonl') if (p.parent / 'sample_submission').is_dir() and (p.parent / 'snapshots').is_dir())
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
arm_dirs = [p.parent for p in Path('/kaggle/input').rglob('agent.yaml') if 'swe_scaffold_v2_b1' in p.read_text(encoding='utf-8')]
assert len(arm_dirs) == 1, f'expected exactly one B1 arm, found {arm_dirs}'
ARM = WORKING_DIR / 'arm'; shutil.rmtree(ARM, ignore_errors=True); shutil.copytree(arm_dirs[0], ARM)
for _rel in ('agent.yaml', 'sub_agents/code_analyzer.yaml'):   # one variant for every agent and sub-agent
    _p = ARM / _rel; _t = _p.read_text(encoding='utf-8'); assert _t.count('gemma-4-31b-it-qat-w4a16-ct') == 1, _rel; _p.write_text(_t.replace('gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct'), encoding='utf-8', newline='\n')
assert 'gemma-4-31b-it-qat-w4a16-ct' not in (ARM / 'agent.yaml').read_text()
(ARM / 'eval_config.yaml').write_text('# B2 (2026-10-20): budget candidate. 8 minutes / 28 tool calls (the harness states both in the task message), 150 s single-command timeout.\n# Why: public teams at 0.22-0.24 report 8 min / 28-30 calls (forum 750496) against our 5 min / no call cap (0.13-0.15); a tool-call budget also bounds the 12 h total:\n# 28 calls x ~9 s (sequential scorer, thinking on) ~ 4.2 min + overhead ~ 5 min per task ~ 10 h for ~120 tasks; exceeding 12 h errors the whole submission.\nevaluation:\n  timeout_seconds: 150\n  max_tool_calls: 28\n  max_time_minutes: 8\n', encoding='utf-8', newline='\n')   # B2 budget: 8 min / 28 tool calls / 150 s command timeout
for must in ('agent.yaml', 'prompts/system.md', 'prompts/analyzer.md', 'configs/sampling.yaml', 'sub_agents/code_analyzer.yaml', 'eval_config.yaml'):
    assert (ARM / must).is_file(), f'arm missing {must}'
print('arm:', arm_dirs[0], '|', (ARM / 'configs/sampling.yaml').read_text().replace(chr(10), ' ')[-120:])
from swegemma.models import load_tasks
ALL_IDS = ["fastapi_11194", "fastapi_13537", "fastapi_13920", "fastapi_14077", "fastapi_14258", "fastapi_14301", "fastapi_14306", "fastapi_14372", "fastapi_14419", "fastapi_14448", "fastapi_14458", "fastapi_14463", "fastapi_14479", "fastapi_14487", "fastapi_14492", "fastapi_14616", "fastapi_14786", "fastapi_14794", "fastapi_14851", "fastapi_14873", "fastapi_14986", "fastapi_15280", "fastapi_15588", "fastapi_15589", "fastapi_15661", "fastapi_15800", "fastapi_5077", "fastapi_5624", "fastapi_9425", "fastapi_9555", "fastapi_9753", "requests_6644", "rich_2725", "rich_2943", "rich_3006", "rich_3043", "rich_3052", "rich_3061", "rich_3063", "rich_3067", "rich_3105", "rich_3180", "rich_3278", "rich_3454", "rich_3468", "rich_3470", "rich_3471", "rich_3472", "rich_3480", "rich_3506", "rich_3518", "rich_3521", "rich_3535", "rich_3675", "rich_3676", "rich_3718", "rich_3772", "rich_3777", "rich_3782", "rich_3882", "rich_3894", "rich_3905", "rich_3930", "rich_3934", "rich_3935", "rich_3938", "rich_3942", "rich_3944", "rich_3953", "rich_4006", "rich_4070", "rich_4075", "rich_4076", "rich_4077", "rich_4079"]
by_id = {t.instance_id: t for t in load_tasks(TASKS_PATH)}
missing = [i for i in ALL_IDS if i not in by_id]
assert not missing, f'task ids missing from the dataset: {missing}'
SCREEN_IDS = random.Random(77).sample(ALL_IDS, len(ALL_IDS))   # all 75 competition tasks, shuffled
assert len(SCREEN_IDS) == 75 and all(i in by_id for i in SCREEN_IDS)
SMOKE_IDS = random.Random(6).sample(ALL_IDS, 2)
print(len(ALL_IDS), 'tasks;', 'smoke ids:', SMOKE_IDS)

## vLLM (real 31B, no adapter) + capture proxy

In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct/2')
assert MODEL_PATH.is_dir(), MODEL_PATH
first_arm = ARM
try:   # swegemma >= 0.2.11 / adk_submission >= 0.2.13 (wheelhouse of 2026-10-09)
    from adk_submission import discover_declared_models
    _declared = discover_declared_models(first_arm); assert len(_declared) == 1, f'expected exactly one declared model, got {_declared}'
    declared_model = next(iter(_declared))
except ImportError:   # older wheelhouse
    from swegemma.models.discovery import validate_single_declared_model
    declared_model = validate_single_declared_model(first_arm)
adapters = discover_adapters(str(first_arm), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=False,
    tensor_parallel_size=tp_size, startup_timeout=60 * 12)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
import capture_proxy as CP
VLLM_URL = f'http://127.0.0.1:{server_instance.config.port}'
proxy = CP.CaptureProxy(VLLM_URL, str(G2 / 'capture.jsonl.gz'), port=8001).start()
server_instance.config.port = 8001      # the model registry reads base_url at creation time: from now on the harness talks to the capture proxy
models = server_instance.create_model_registry(aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')
print('harness endpoint (proxy):', server_instance.base_url, '-> upstream', VLLM_URL)

## Smoke round, capture gate, full rounds

In [ ]:
ARMS_SRC = "\"\"\"Config-screen arms (G6): copies of the B1 arm whose only difference is configs/sampling.yaml.\n\nT1: shorter thinking budget and output cap (cut the long turns that eat the 5-minute clock)\nT2: T1 + Gemma's usual sampling (temperature 1.0, top_p 0.95, top_k 64)\n\"\"\"\n\nimport re\nimport shutil\nfrom pathlib import Path\n\nARMS = {\n    \"t1\": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=4096, thinking_budget=2048),\n    \"t2\": dict(temperature=1.0, top_p=0.95, top_k=64, max_output_tokens=4096, thinking_budget=2048),\n    \"t4\": dict(temperature=1.0, top_p=0.95, top_k=64, max_output_tokens=8192, thinking_budget=4096),   # G13 probe: B1 settings except Gemma's usual sampling\n    \"e4b\": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=8192, thinking_budget=4096),   # G14a: E4B baseline with the B1/B2 sampling\n    \"e4b_b2\": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=8192, thinking_budget=4096),   # G14c: E4B, B2 prompt (budget-agnostic)\n    \"e4b_v\": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=8192, thinking_budget=4096),   # G14c: E4B, B2 prompt + reproduce-then-verify schedule\n    \"t3\": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=3072, thinking_budget=1024),   # G12 (31B): half of T1's thinking budget\n}\n\n\ndef sampling_yaml(temperature, top_p, top_k, max_output_tokens, thinking_budget) -> str:\n    lines = [f\"temperature: {temperature}\", f\"top_p: {top_p}\"]\n    if top_k is not None:\n        lines.append(f\"top_k: {top_k}\")\n    lines += [f\"max_output_tokens: {max_output_tokens}\", \"thinking_config:\", f\"  thinking_budget: {thinking_budget}\", \"  include_thoughts: true\", \"\"]\n    return \"\\n\".join(lines)\n\n\ndef make_arm(src: Path, dst: Path, **kw) -> Path:\n    shutil.rmtree(dst, ignore_errors=True)\n    shutil.copytree(src, dst)\n    p = dst / \"configs\" / \"sampling.yaml\"\n    old = p.read_text(encoding=\"utf-8\")\n    assert re.search(r\"^temperature:\", old, re.M) and \"thinking_budget\" in old, \"unexpected sampling.yaml layout\"\n    p.write_text(sampling_yaml(**kw), encoding=\"utf-8\", newline=\"\\n\")\n    return dst\n\n\ndef request_matches(request: dict, expect: dict) -> list:\n    \"\"\"Compare one captured vLLM request body with the arm's settings; returns the list of mismatches (empty = OK).\"\"\"\n    bad = []\n    if request.get(\"temperature\") != expect[\"temperature\"]:\n        bad.append((\"temperature\", request.get(\"temperature\"), expect[\"temperature\"]))\n    if request.get(\"top_p\") != expect[\"top_p\"]:\n        bad.append((\"top_p\", request.get(\"top_p\"), expect[\"top_p\"]))\n    if expect.get(\"top_k\") is not None and request.get(\"top_k\") != expect[\"top_k\"]:\n        bad.append((\"top_k\", request.get(\"top_k\"), expect[\"top_k\"]))\n    mo = request.get(\"max_completion_tokens\", request.get(\"max_tokens\"))\n    if mo != expect[\"max_output_tokens\"]:\n        bad.append((\"max_output_tokens\", mo, expect[\"max_output_tokens\"]))\n    if request.get(\"thinking_token_budget\") != expect[\"thinking_budget\"]:\n        bad.append((\"thinking_token_budget\", request.get(\"thinking_token_budget\"), expect[\"thinking_budget\"]))\n    return bad\n"
import asyncio, concurrent.futures, requests, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_fn):
    try: loop = asyncio.get_running_loop()
    except RuntimeError: loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn())).result()
    return asyncio.run(coro_fn())

def budget_of(arm_dir):
    cfg = yaml.safe_load((arm_dir / 'eval_config.yaml').read_text(encoding='utf-8')).get('evaluation', {})
    out = {}
    for k, cast in (('timeout_seconds', int), ('max_tool_calls', int), ('max_time_minutes', float)):
        if k in cfg: out[k] = cast(cfg[k])
    t = cfg.get('max_turns', cfg.get('max_llm_calls'))
    if t is not None: out['max_turns'] = int(t)
    return out

limits, gen_constraints = build_submission_limits()
print('per-task budget:', budget_of(ARM))

def run_round(tag, ids, arm_dir=None):
    arm_dir = arm_dir or ARM
    proxy.round_tag = tag
    rd = G2 / tag
    cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=rd, submission_dir=arm_dir, models=models, sandbox='subprocess',
        task_ids=list(ids), concurrency=4, display_mode='quiet', limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'), wheels_dir=DATA_DIR / 'wheels', verbose=False, **budget_of(arm_dir))
    t0 = time.perf_counter()
    try:
        run_sync(Evaluator(cfg).run)
    except Exception:
        traceback.print_exc(); RESULTS[tag] = {'error': 'run raised', 'trace': traceback.format_exc()[-1500:]}; save_results(); return None
    rows = [json.loads(l) for l in open(rd / 'task_results.jsonl', encoding='utf-8')]
    RESULTS[tag] = {'n': len(rows), 'resolved': sum(r['resolved'] for r in rows), 'patch_written': sum(1 for r in rows if r['agent_patch_size'] > 0),
                    'time_cap_hit': sum(1 for r in rows if 'session timeout' in (r['error'] or '')), 'wall_s': round(time.perf_counter() - t0, 1),
                    'mean_task_s': round(sum(r['duration_seconds'] for r in rows) / max(1, len(rows)), 1), 'resolved_ids': sorted(r['instance_id'] for r in rows if r['resolved']),
                    'proxy': proxy.stats()}
    save_results(); print(tag, {k: v for k, v in RESULTS[tag].items() if k != 'resolved_ids'}, flush=True)
    return rows

# ------------------------------------------------------------------ G14c: E4B prompt arms over all 75 tasks, each preceded by a checked 2-task smoke
exec(ARMS_SRC)   # defines ARMS, make_arm, request_matches
ARM_DIRS = {n: make_arm(ARM, WORKING_DIR / f'arm_{n}', **ARMS[n]) for n in ('e4b_b2', 'e4b_v')}
PROMPTS = {'e4b_b2': 'You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\n\n## Core Objective: Fast, Minimal, and Precise Fixes\nYou have a HARD budget for the whole task, stated in the Task Budget block of the task message (wall-clock time and number of tool calls). The session ends when either is exhausted, and whatever is in the working tree is graded as-is, so a plausible partial fix beats no patch, and never revert a plausible fix just because you are unsure. Plan by phases of that allowance and follow this schedule:\n1. Locate (at most 4 tool calls): grep for exact identifiers or error text from the issue, then read the one function the issue is about.\n2. **Make your FIRST EDIT with `edit_file` by your 8th tool call**, using your best hypothesis, even if you are not fully certain. Do not keep exploring once you have found the function the issue is about.\n3. Verify with one targeted test (output redirected to a file, then `tail` it) and refine only if it fails.\n4. Call `submit_patch` as soon as the targeted test passes, or as soon as the budget is running low (about 4 tool calls or one minute left).\n\n## Workflow\n\n### 1. Locate the Target Code\n- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement, and `grep` for them first: `grep -rn "<literal identifier or error text>" --include=*.py .` This is usually faster and more reliable than the graph tools below.\n- **`get_code_neighbors(node)` is an exact-symbol lookup, NOT free-text/semantic search.** `node` must be an EXACT fully-qualified symbol already in the code graph (e.g. `package.module.ClassName.method_name`) — a natural-language description will always return zero results.\n- **The code graph has exactly ONE edge relation: `calls` (lowercase).** Omit `edge_type` or pass exactly `"calls"`. Do not invent other edge types (e.g. `CALLS`, `DEFINED_IN`, `IMPORTS`) — they do not exist in this data and silently return zero results even for a valid node.\n- **The code graph has NO async functions or methods at all.** For async-heavy code (common in e.g. FastAPI request-handling paths), `get_code_neighbors`/`get_code_subgraph` cannot see most of the relevant call chain — go straight to `grep`/`read_file` instead of trying graph tools first.\n- Preferred order:\n  1. `grep` for exact identifiers/error text from the problem statement.\n  2. Once you have an exact symbol name, `get_code_neighbors(node)` to see its callers/callees, then `get_code_subgraph(nodes)` if you need to see how several related symbols connect — but skip this entirely for async code.\n  3. `read_file` on the specific file/line range once you know exactly what to read.\n\n### 2. Implement the Solution Directly\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\n- **Keep each `edit_file`/`write_file` call focused and small.** A very large edit combined with a large thinking block can hit the output token limit before the tool call finishes closing, which corrupts the turn. If a change is large, split it into several smaller, incremental edits rather than one big one.\n\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\n- **`run_command` output is truncated to its first 5,000 characters, but pytest prints its pass/fail verdict at the END.** Running pytest directly can silently cut off the one line that tells you whether it passed. Always redirect to a file and read the end first: `pytest tests/test_target.py -k test_feature > /tmp/test_1.log 2>&1; tail -n 30 /tmp/test_1.log`. Use a new file name each time (`_1`, `_2`, ...) if you re-run. The same applies to any command that might produce long output (big greps, tracebacks).\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures, or environment import errors unrelated to your task.\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes and exhaust your turn budget.\n- If you need to locate the test file, find it explicitly with `find tests -name "*<name>*.py"` instead of running the test runner across the repo.\n- If you need a scratch/reproduction script to check your understanding of the bug, write it to `/tmp/repro.py` (via `run_command`), never under `/workspace` — the file tools (`read_file`/`write_file`/`edit_file`) only accept paths inside `/workspace` and will fail on `/tmp` paths. Anything left inside `/workspace` when you call `submit_patch` — including scratch scripts — becomes part of your submitted patch.\n- If the same command or tool call fails twice in a row, change approach rather than retrying it verbatim (e.g. if `read_file` fails, try `sed -n \'START,ENDp\' <file>` via `run_command`).\n\n### 4. Immediate Patch Submission\n- Once your targeted test passes:\n  1. Double check you have not left any scratch files under `/workspace` and have not modified `/workspace/pytest.ini` or `/workspace/conftest.py`.\n  2. Call `submit_patch` immediately. It does not count against your tool-call budget, so there is no cost to calling it as soon as you\'re done — but the session ends as soon as you do, so call it last.\n  3. Verify `patch_size > 0` and `files_changed > 0`.\n  4. Output a short summary of the fix to end the session.\n- Call `get_status` every ~8 tool calls (it is free, does not count against your budget). If it shows a `budget_warning`, stop exploring immediately and submit whatever correct, minimal fix you have rather than risk running out of budget with nothing submitted.\n\n## Anti-Patterns to Avoid\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\n- **NEVER modify `/workspace/pytest.ini` or `/workspace/conftest.py`** — these are part of the harness\'s baseline commit; changing them shows up in your diff and can break verification.\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) — always specify the exact test file path.\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** — your task is strictly to implement the fix for the reported issue in source code.\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\n- Do NOT refactor or reformat unrelated functions or files.\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\n', 'e4b_v': 'You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\n\n## Core Objective: Fast, Minimal, and Precise Fixes\nYou have a HARD budget for the whole task, stated in the Task Budget block of the task message (wall-clock time and number of tool calls). The session ends when either is exhausted, and whatever is in the working tree is graded as-is, so a plausible partial fix beats no patch, and never revert a plausible fix just because you are unsure. Plan by phases of that allowance and follow this schedule:\n1. Locate (at most 6 tool calls): grep for exact identifiers or error text from the issue, read the function the issue is about, and find the existing test file for that module (`find tests -name "*<name>*.py"`) and read the tests nearest to the behaviour.\n2. Reproduce (1-2 tool calls): write `/tmp/repro.py` that exercises exactly the behaviour the issue describes (use the examples in the issue), run it and confirm that it fails the way the issue says. If a reproduction is impractical, skip this step.\n3. **Make your FIRST EDIT with `edit_file` by your 10th tool call**, using your best hypothesis. Keep the change minimal.\n4. Verify (this is the most important step; spend 5-8 tool calls on it): (a) re-run `/tmp/repro.py` and confirm that it now passes; (b) run the targeted existing tests for the module you changed (redirect to a file, then `tail`); (c) run `git diff` and compare it line by line with the issue: does the code now produce the exact expected strings, exception types, status codes, return values and signatures the issue asks for, in EVERY case the issue lists? (d) `grep` for the same buggy pattern elsewhere in the module (a second place with the same bug is common). If anything is off, fix it and verify again.\n5. Call `submit_patch` only after the checks in step 4 pass, or as soon as the budget is running low (about 4 tool calls or one minute left). A plausible fix always beats no patch.\n\n## Workflow\n\n### 1. Locate the Target Code\n- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement, and `grep` for them first: `grep -rn "<literal identifier or error text>" --include=*.py .` This is usually faster and more reliable than the graph tools below.\n- **`get_code_neighbors(node)` is an exact-symbol lookup, NOT free-text/semantic search.** `node` must be an EXACT fully-qualified symbol already in the code graph (e.g. `package.module.ClassName.method_name`) — a natural-language description will always return zero results.\n- **The code graph has exactly ONE edge relation: `calls` (lowercase).** Omit `edge_type` or pass exactly `"calls"`. Do not invent other edge types (e.g. `CALLS`, `DEFINED_IN`, `IMPORTS`) — they do not exist in this data and silently return zero results even for a valid node.\n- **The code graph has NO async functions or methods at all.** For async-heavy code (common in e.g. FastAPI request-handling paths), `get_code_neighbors`/`get_code_subgraph` cannot see most of the relevant call chain — go straight to `grep`/`read_file` instead of trying graph tools first.\n- Preferred order:\n  1. `grep` for exact identifiers/error text from the problem statement.\n  2. Once you have an exact symbol name, `get_code_neighbors(node)` to see its callers/callees, then `get_code_subgraph(nodes)` if you need to see how several related symbols connect — but skip this entirely for async code.\n  3. `read_file` on the specific file/line range once you know exactly what to read.\n\n### 2. Implement the Solution Directly\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\n- **Keep each `edit_file`/`write_file` call focused and small.** A very large edit combined with a large thinking block can hit the output token limit before the tool call finishes closing, which corrupts the turn. If a change is large, split it into several smaller, incremental edits rather than one big one.\n\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\n- **`run_command` output is truncated to its first 5,000 characters, but pytest prints its pass/fail verdict at the END.** Running pytest directly can silently cut off the one line that tells you whether it passed. Always redirect to a file and read the end first: `pytest tests/test_target.py -k test_feature > /tmp/test_1.log 2>&1; tail -n 30 /tmp/test_1.log`. Use a new file name each time (`_1`, `_2`, ...) if you re-run. The same applies to any command that might produce long output (big greps, tracebacks).\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures, or environment import errors unrelated to your task.\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes and exhaust your turn budget.\n- If you need to locate the test file, find it explicitly with `find tests -name "*<name>*.py"` instead of running the test runner across the repo.\n- If you need a scratch/reproduction script to check your understanding of the bug, write it to `/tmp/repro.py` (via `run_command`), never under `/workspace` — the file tools (`read_file`/`write_file`/`edit_file`) only accept paths inside `/workspace` and will fail on `/tmp` paths. Anything left inside `/workspace` when you call `submit_patch` — including scratch scripts — becomes part of your submitted patch.\n- If the same command or tool call fails twice in a row, change approach rather than retrying it verbatim (e.g. if `read_file` fails, try `sed -n \'START,ENDp\' <file>` via `run_command`).\n\n### 4. Immediate Patch Submission\n- Once your targeted test passes AND the step 4 review of `git diff` finds nothing missing:\n  1. Double check you have not left any scratch files under `/workspace` and have not modified `/workspace/pytest.ini` or `/workspace/conftest.py`.\n  2. Call `submit_patch` immediately. It does not count against your tool-call budget, so there is no cost to calling it as soon as you\'re done — but the session ends as soon as you do, so call it last.\n  3. Verify `patch_size > 0` and `files_changed > 0`.\n  4. Output a short summary of the fix to end the session.\n- Call `get_status` every ~8 tool calls (it is free, does not count against your budget). If it shows a `budget_warning`, stop exploring immediately and submit whatever correct, minimal fix you have rather than risk running out of budget with nothing submitted.\n\n## Anti-Patterns to Avoid\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\n- **NEVER modify `/workspace/pytest.ini` or `/workspace/conftest.py`** — these are part of the harness\'s baseline commit; changing them shows up in your diff and can break verification.\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) — always specify the exact test file path.\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** — your task is strictly to implement the fix for the reported issue in source code.\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\n- Do NOT refactor or reformat unrelated functions or files.\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\n'}
for n, d in ARM_DIRS.items():
    assert n in PROMPTS, n
    (d / 'prompts' / 'system.md').write_text(PROMPTS[n], encoding='utf-8', newline=chr(10))
    assert (d / 'prompts' / 'system.md').read_text().startswith('You are an expert autonomous software engineer')
    print(n, 'system prompt chars', len(PROMPTS[n]))
for n, d in ARM_DIRS.items(): print(n, ARMS[n], '|', (d / 'configs/sampling.yaml').read_text().replace(chr(10), ' | '))
MAIN = 'You are an expert autonomous software engineer'

def config_gate(tag, arm):
    reqs = []
    for rec in CP.read_records(str(G2 / 'capture.jsonl.gz')):
        if rec['round'] != tag: continue
        sysm = next((m.get('content') for m in rec['request']['messages'] if m.get('role') == 'system'), '') or ''
        if sysm.startswith(MAIN): reqs.append(rec['request'])
    bad = [m for r in reqs for m in request_matches(r, ARMS[arm])]
    RESULTS['config_gate_' + tag] = {'main_agent_requests': len(reqs), 'mismatches': bad[:5], 'n_mismatches': len(bad)}
    save_results(); print('config gate', tag, RESULTS['config_gate_' + tag])
    if not reqs or bad:
        raise RuntimeError(f'CONFIG GATE FAILED for arm {arm}: {len(reqs)} requests, mismatches {bad[:3]}')

for k, arm in enumerate(('e4b_b2', 'e4b_v')):
    elapsed = (time.time() - T_START) / 60
    if k > 0 and elapsed + 100 > BUDGET_MIN - 5:
        print(f'not starting {arm}: elapsed {elapsed:.0f} min + ~100 min would exceed the budget'); break
    rows = run_round(f'smoke_{arm}', SMOKE_IDS, ARM_DIRS[arm])
    if k == 0:
        st = proxy.stats(); print('capture after smoke:', st)
        assert rows and st['records'] >= 6 and st['errors'] == 0, f'capture/serving problem after smoke: {st}'
        try:
            first = next(CP.read_records(str(G2 / 'capture.jsonl.gz'))); rq = first['request']
            tk = {kk: rq[kk] for kk in ('model', 'messages', 'tools', 'chat_template_kwargs') if kk in rq}; tk['add_generation_prompt'] = True
            tok = requests.post(VLLM_URL + '/tokenize', json=tk, timeout=300).json().get('tokens')
            got = first['response'].get('prompt_token_ids')
            RESULTS['token_id_crosscheck'] = {'tokenize_len': len(tok or []), 'captured_len': len(got or []), 'identical': tok == got}
        except Exception:
            RESULTS['token_id_crosscheck'] = {'error': traceback.format_exc()[-600:]}
        print('cross-check:', RESULTS['token_id_crosscheck'])
    config_gate(f'smoke_{arm}', arm)
    run_round(arm, SCREEN_IDS, ARM_DIRS[arm])

# ------------------------------------------------------------------ mechanism summary from the capture (main-agent requests only)
import collections, statistics as _st
mech = collections.defaultdict(lambda: {'turns': 0, 'sec': 0.0, 'long_sec': 0.0, 'long_turns': 0, 'ct': [], 'length': 0})
for rec in CP.read_records(str(G2 / 'capture.jsonl.gz')):
    if rec['round'] not in ('e4b_b2', 'e4b_v'): continue
    sysm = next((m.get('content') for m in rec['request']['messages'] if m.get('role') == 'system'), '') or ''
    if not sysm.startswith(MAIN): continue
    ct = (rec['response'].get('usage') or {}).get('completion_tokens', 0); dt = rec['t1'] - rec['t0']
    m = mech[rec['round']]; m['turns'] += 1; m['sec'] += dt; m['ct'].append(ct)
    if ct > 2000: m['long_turns'] += 1; m['long_sec'] += dt
    if rec['response']['choices'][0].get('finish_reason') == 'length': m['length'] += 1
RESULTS['mechanism'] = {r: {'turns': m['turns'], 'mean_completion_tokens': round(_st.mean(m['ct']), 1), 'turns_over_2000': m['long_turns'], 'length_finishes': m['length'],
                            'time_share_in_turns_over_2000': round(m['long_sec'] / max(m['sec'], 1e-9), 3), 'mean_request_s': round(m['sec'] / max(m['turns'], 1), 2)} for r, m in mech.items()}
print('mechanism:', json.dumps(RESULTS['mechanism'], indent=1))
print('\n===== G14c (E4B prompt arms) SUMMARY =====')
for tag in [f'{p}{a}' for a in ('e4b_b2', 'e4b_v') for p in ('smoke_', '')]:
    if tag in RESULTS and 'n' in RESULTS[tag]: print(tag, {kk: v for kk, v in RESULTS[tag].items() if kk not in ('resolved_ids', 'proxy')})
print('capture:', proxy.stats(), '| file MB:', round((G2 / 'capture.jsonl.gz').stat().st_size / 1e6, 1))
RESULTS['finished'] = time.strftime('%Y-%m-%d %H:%M:%S'); RESULTS['capture'] = proxy.stats(); save_results()
_wd.cancel()